# NetworkHealth AI — Phase 1/2 training on a Colab T4

This notebook is the **GPU-only** path. It imports the shared core from
`networkhealth/` unchanged, so the features trained here are byte-identical to
the features the laptop computes at inference time.

## Open this in VS Code with the Colab extension

1. Install the **Colab** extension in VS Code (Google's official one).
2. Open this file, then pick a runtime: *Select Kernel → Colab → T4 GPU*.
3. Run the cells top to bottom. The extension executes on the Colab VM while
   the notebook file stays in this repo.

## Why the data is prepared locally, not here

Collection and cleaning run on your machine (or in cell 2, if you prefer
Colab's bandwidth). Training then consumes `features.npz`, which is already a
contiguous float32 array — so no CSV parsing or cleaning happens inside the
training loop. That is the single biggest throughput win on a T4.

## Honest note on dataset size

Public RIPE Atlas ping data yields roughly **one window per observation**. To
reach 100M windows you would need ~110M raw observations. A wide Atlas pull
plus authorized local telemetry realistically gives **1–5M windows**. Cell 4
reports the actual count; treat that as the real scale, not a target to inflate.

## Security scope

This notebook trains **performance** heads only: RTT, jitter, loss,
degradation, anomaly suspicion. It does **not** train a spoofing/attack
classifier — that needs labelled positives from authorized local telemetry and
is kept separate on purpose (Phase 6). An unusual RTT is not an attack.

## Per-Instance Identity Masking

Telemetry features and model representations are decoupled from client identity:
1. **Cryptographic Pseudonyms & Micro-Jitter**: Raw slot/probe IDs are HMAC-hashed, and timestamps receive +/-125ms micro-jitter to prevent clock-phase correlation across co-located client instances.
2. **Relative Baseline Tracking**: Latencies are tracked via EWMA to compute relative deltas and robust z-scores, preventing the model from overfitting to static proxy RTT offsets.
3. **Neural Instance Normalization**: An `InstanceIdentityMaskLayer` (`nn.InstanceNorm1d`) inside `NetworkHealthTCN` enforces instance-invariant temporal feature representations before the TCN residual blocks.


## 1. Environment

Confirm the T4 and the installed PyTorch build before spending quota.

In [1]:
import sys, platform, subprocess
print('python', sys.version.split()[0], '|', platform.platform())
try:
    import torch
    print('torch', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('sm count:', props.multi_processor_count,
              '| total mem GB:', round(props.total_memory / 2**30, 2))
        print('bfloat16 supported:', props.major >= 8)
        print('=> T4 (sm75) has FP16 Tensor Cores but NO bfloat16.')
except ImportError:
    print('torch missing; install with the cell below')
print('numpy', __import__('numpy').__version__)


python 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
torch 2.11.0+cu130
cuda available: True
gpu: Tesla T4
sm count: 40 | total mem GB: 14.56
bfloat16 supported: False
=> T4 (sm75) has FP16 Tensor Cores but NO bfloat16.
numpy 2.1.3


In [2]:
# Only run if torch is missing.
# Colab's preinstalled torch is usually the right build for T4; prefer it.
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu121
pass

## 2. Get the repository and the prepared features

Two options: mount Drive (recommended, keeps artifacts across sessions),
or upload `features.npz` directly.

In [3]:
# Mount Drive. We only use it to persist artifacts across sessions.
# The repo itself is cloned into /content below, which trains much faster
# than reading from Drive (Drive I/O would bottleneck the data loader).
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
    print('Drive mounted:', DRIVE.exists())
except Exception as exc:
    DRIVE = None
    print('Drive not mounted; artifacts will not persist:', exc)


Mounted at /content/drive
Drive not mounted; artifacts will not persist: name 'Path' is not defined


In [4]:
# Clone the repo into the VM. This is the path that actually matters.
import shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/saoxdxd2/dofus-emu.git'
CLONE_DIR = Path('/content/dofus-emu')
REPO_DIR = CLONE_DIR / 'dofus-network'

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

result = subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)],
    capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())
print('clone exit code:', result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        'git clone failed. If the repo is private, either make it public '
        'temporarily or use a personal access token in the URL:\n'
        '  https://<token>@github.com/saoxdxd2/dofus-emu.git'
    )

# Verify the package arrived, not just the notebook.
pkg = REPO_DIR / 'networkhealth' / '__init__.py'
print('package present :', pkg.exists())
if not pkg.exists():
    listing = sorted(p.name for p in CLONE_DIR.iterdir())
    raise FileNotFoundError(
        f'Clone succeeded but {pkg} is missing.\n'
        f'Contents of {CLONE_DIR}: {listing}'
    )


Cloning into '/content/dofus-emu'...
clone exit code: 0
package present : True


In [5]:
import importlib, sys
from pathlib import Path

# REPO_DIR should have been set by the clone cell above. Guard against a
# stale value left in the kernel by an older version of the mount cell,
# which pointed REPO_DIR at a Drive path that may not exist.
try:
    REPO_DIR
except NameError:
    raise NameError(
        'REPO_DIR is not defined. Run the clone cell above first.'
    )

REPO_DIR = Path(REPO_DIR)
pkg_init = REPO_DIR / 'networkhealth' / '__init__.py'

print('REPO_DIR         :', REPO_DIR)
print('REPO_DIR exists  :', REPO_DIR.exists())
print('package present  :', pkg_init.exists())

if not pkg_init.exists():
    # Diagnose instead of failing with a bare ModuleNotFoundError.
    if not REPO_DIR.exists():
        hint = (
            'REPO_DIR does not exist. The clone cell probably did not run, '
            'or it set REPO_DIR to a different path. Re-run the clone cell.'
        )
    else:
        contents = sorted(p.name for p in REPO_DIR.iterdir())
        hint = (
            f'REPO_DIR exists but networkhealth/__init__.py is missing.\n'
            f'Contents: {contents}\n'
            'If the listing above shows no networkhealth folder, the clone '
            'pulled an incomplete tree or the package was never committed. '
            'Run `git ls-files` in the clone to confirm.'
        )
    raise FileNotFoundError(hint)

FEATURES = REPO_DIR / 'networkhealth_artifacts' / 'features.npz'
OUT_DIR = REPO_DIR / 'networkhealth_artifacts'

# features.npz is gitignored, so a fresh clone does NOT include it.
# Try Drive first if a copy was uploaded there.
if not FEATURES.exists():
    candidates = []
    if 'DRIVE' in dir() and DRIVE is not None:
        candidates += [
            DRIVE / 'dofus-network' / 'networkhealth_artifacts' / 'features.npz',
            DRIVE / 'dofus-emu' / 'dofus-network' / 'networkhealth_artifacts' / 'features.npz',
            DRIVE / 'networkhealth_artifacts' / 'features.npz',
            DRIVE / 'features.npz',
        ]
    for candidate in candidates:
        if candidate.exists():
            OUT_DIR.mkdir(parents=True, exist_ok=True)
            import shutil
            shutil.copy2(candidate, FEATURES)
            print('copied features from Drive:', candidate)
            break

print('features present :', FEATURES.exists())

# Drop any stale sys.path entries so an old repo location cannot shadow this one.
repo = str(REPO_DIR)
sys.path[:] = [p for p in sys.path if 'dofus-network' not in p and 'dofus-emu' not in p]
sys.path.insert(0, repo)

importlib.invalidate_caches()
import networkhealth
print('networkhealth', networkhealth.__version__)
print('imported from    :', networkhealth.__file__)
from networkhealth.features import FEATURE_NAMES, HORIZONS
print('features:', len(FEATURE_NAMES), FEATURE_NAMES)
print('horizons (observations ahead):', HORIZONS)


REPO_DIR         : /content/dofus-emu/dofus-network
REPO_DIR exists  : True
package present  : True
features present : False
networkhealth 0.2.0
imported from    : /content/dofus-emu/dofus-network/networkhealth/__init__.py
features: 10 ('log_rtt_ms', 'is_missing', 'log_gap_seconds', 'log_rtt_delta', 'loss_rate_8', 'loss_rate_32', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos')
horizons (observations ahead): (1, 3, 6)


In [6]:
# Collect and prepare REAL RIPE Atlas data on this VM.
#
# Output streams live. The previous version used capture_output=True, which
# buffered everything until the process exited - that is why the cell looked
# hung for minutes with no feedback.
#
# Timings: collection is dominated by polite API pacing in the collector
# (sleep between requests). 25 measurements typically takes several minutes,
# and can be longer if the API rate-limits us. Start with fewer measurements
# to confirm the pipeline works before running a large pull.

import subprocess, sys
from pathlib import Path

MEASUREMENTS = 25
DAYS = 7

def run(step, args):
    print('=' * 70, flush=True)
    print('STEP:', step, flush=True)
    print('=' * 70, flush=True)
    # -u disables Python's own output buffering in the child process, so the
    # collector's progress lines appear as they happen.
    cmd = [sys.executable, '-u', '-m', 'networkhealth', *args]
    print('$', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    if result.returncode != 0:
        raise RuntimeError(f'{step} failed with exit code {result.returncode}')
    print(flush=True)

run('collect', ['collect', '--days', str(DAYS),
                '--measurements', str(MEASUREMENTS)])
run('clean',   ['clean', '--input', 'networkhealth_artifacts/raw.jsonl'])
run('prepare', ['prepare', '--input', 'networkhealth_artifacts/clean.jsonl', '--mask-identity'])

features = REPO_DIR / 'networkhealth_artifacts' / 'features.npz'
print('features.npz present:', features.exists(), flush=True)
if features.exists():
    print('size MB:', round(features.stat().st_size / 2**20, 2), flush=True)


STEP: collect
$ /usr/bin/python3 -u -m networkhealth collect --days 7 --measurements 25


KeyboardInterrupt: 

In [ ]:
# Verify features exist now that collection has had a chance to run.
features = REPO_DIR / 'networkhealth_artifacts' / 'features.npz'
FEATURES = features

print('features.npz present:', features.exists())
if not features.exists():
    raise FileNotFoundError(
        'features.npz still missing after the collect/prepare cell.\n'
        'Check that cell for errors. Common causes:\n'
        '  - RIPE Atlas API rate limiting (wait a few minutes and retry)\n'
        '  - too few measurements discovered (try --measurements 50)\n'
        '  - too few observations to build windows (increase --days)'
    )

import numpy as np
payload = np.load(features, allow_pickle=False)
print('windows:', f"{payload['x'].shape[0]:,}")
print('seq_len:', payload['x'].shape[1], '| features:', payload['x'].shape[2])


## 3. Inspect the dataset before training

Know your window count and the split you will get, *before* the GPU bill.

In [ ]:
import numpy as np
payload = np.load(FEATURES, allow_pickle=False)
x, y = payload['x'], payload['y']
print('windows:', f'{x.shape[0]:,}')
print('seq_len:', x.shape[1], '| features:', x.shape[2])
print('array bytes:', round(x.nbytes / 2**20, 1), 'MB  (contiguous, pre-cleaned)')
print('distinct series:', len(set(payload['series_index'].tolist())))
print('distinct probes:', len(set(payload['probe_ids'].tolist())))
block = y.reshape(y.shape[0], len(HORIZONS), 5)
print('loss rate per horizon:', [round(float(block[:, i, 4].mean()), 4) for i in range(len(HORIZONS))])
print('valid RTT targets per horizon:', [int((block[:, i, 1] > 0.5).sum()) for i in range(len(HORIZONS))])


## 4. Batch-size sweep

Measure samples/second at 64/128/256/512. The T4's Tensor Cores need
batch ≥ ~128 to pay off, but bigger is not automatically better — this
records the evidence for whichever size you pick.

In [ ]:
from colab.train_gpu import GpuTrainConfig, load_windows, probe_batch_sizes
from networkhealth.model import ModelConfig, NetworkHealthTCN

cfg = GpuTrainConfig()
data = load_windows(FEATURES, cfg.seq_len)
probe_model = NetworkHealthTCN(ModelConfig(hidden=cfg.hidden, levels=cfg.levels,
                                          dropout=cfg.dropout, horizons=HORIZONS,
                                          use_identity_mask=cfg.use_identity_mask)).cuda()
print('identity mask active:', cfg.use_identity_mask)
print('parameters:', f'{probe_model.parameter_count():,}')
sweep = probe_batch_sizes(probe_model, data['x'], amp=cfg.amp)
for row in sweep.results:
    if 'error' in row:
        print(f"  batch {row['batch_size']:>4}: FAILED ({row['error'][:60]}...)")
    else:
        print(f"  batch {row['batch_size']:>4}: {row['samples_per_second']:>9,.1f} samples/s "
              f"| peak {row['peak_memory_mb']:>7,.1f} MB")
best = sweep.best()
print('best batch size by throughput:', best['batch_size'] if best else 'n/a')


## 5. Train

FP16 AMP with gradient scaling; loss and probability heads forced to FP32.
The driver runs its own sweep internally and records the chosen batch size.

In [ ]:
from colab.train_gpu import train_on_gpu

artifacts = train_on_gpu(FEATURES, cfg, OUT_DIR)
print()
print('checkpoint:', artifacts['checkpoint'])
print('best valid loss:', artifacts['checkpoint'])
print('chosen batch size:', artifacts['chosen_batch_size'])
print('wall clock (s):', artifacts['wall_clock_seconds'])


## 6. Results and honest comparison

Read the per-horizon numbers, then compare against the baselines computed
on the *same* validation rows. If the model does not beat persistence,
that is the result — do not adopt a more complex model anyway.

In [ ]:
import json
report = json.loads((OUT_DIR / 'evaluation.json').read_text())
cost = report['cost']
print('CPU inference cost (this is what the laptop will pay):')
print('  parameters      :', f"{cost['parameters']:,}")
print('  state dict      :', cost['state_dict_mb'], 'MB')
print('  latency median  :', cost['latency_ms_median'], 'ms')
print('  latency p95     :', cost['latency_ms_p95'], 'ms')
print()
print(f"{'horizon':>8} {'rtt_mae':>9} {'loss_brier':>11} {'loss_ece':>9} {'deg_f1':>8}")
for horizon, block in report['model']['per_horizon'].items():
    rtt_mae = block.get('rtt_log_mae')
    brier = block.get('loss_brier')
    ece = block.get('loss_ece')
    f1 = (block.get('degradation_detection') or {}).get('f1')
    fmt = lambda v: 'n/a' if v is None else f'{v:.4f}'
    print(f"{horizon:>8} {fmt(rtt_mae):>9} {fmt(brier):>11} {fmt(ece):>9} {fmt(f1):>8}")
print()
print('Baselines on the same split (run `python -m networkhealth baselines` locally):')
baseline_path = OUT_DIR / 'baselines.json'
if baseline_path.exists():
    base = json.loads(baseline_path.read_text())
    for name, block in base['baselines'].items():
        row = block['per_horizon'].get(str(HORIZONS[0]), {})
        mae = row.get('rtt_log_mae')
        print(f"  {name:>16}: rtt_log_mae={mae if mae is None else round(mae,4)}")
else:
    print('  baselines.json not found next to the checkpoint.')


## 7. Save artifacts for the local machine

The checkpoint carries the frozen normaliser and feature order, so the
local inference module cannot mis-parse it. Copy `model.pt` and
`evaluation.json` into the repo's `networkhealth_artifacts/`.

In [ ]:
print('Files to copy back to the laptop:')
for name in ('model.pt', 'evaluation.json'):
    path = OUT_DIR / name
    print(f"  {path}  exists={path.exists()}  "
          f"size={round(path.stat().st_size/2**20,2) if path.exists() else 0} MB")
print()
print('Local verification:')
print('  python -m networkhealth.inference --checkpoint networkhealth_artifacts/model.pt')
